In [1]:
import urllib.request
import json

url = "https://api.open-meteo.com/v1/forecast?latitude=13.75&longitude=100.5&hourly=temperature_2m&forecast_days=7&timezone=Asia/Bangkok"

text = urllib.request.urlopen(url, timeout=15).read().decode("utf-8")
data = json.loads(text)

In [2]:
times = data["hourly"]["time"]
temps = data["hourly"]["temperature_2m"]

rows = []
for i in range(len(times)):
    rows.append(["Bangkok", times[i], temps[i]])

rows[:5]


[['Bangkok', '2026-08-28T00:00', 27.1],
 ['Bangkok', '2026-08-28T01:00', 26.9],
 ['Bangkok', '2026-08-28T02:00', 26.9],
 ['Bangkok', '2026-08-28T03:00', 26.8],
 ['Bangkok', '2026-08-28T04:00', 26.5]]

In [3]:
import sqlite3

conn = sqlite3.connect("DB/weather.db")

conn.execute("""
    CREATE TABLE IF NOT EXISTS weather (
        city        TEXT,
        time        TEXT,
        temperature REAL
    )
""")
conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_weather_city_time ON weather(city, time)")
conn.commit()

print("สร้างตารางแล้ว")

สร้างตารางแล้ว


In [4]:
conn.executemany("INSERT OR REPLACE INTO weather VALUES (?, ?, ?)", rows)
conn.commit()

print("ใส่แล้ว", len(rows), "แถว")

conn.close()

ใส่แล้ว 168 แถว


In [6]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.3/818.3 kB 3.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.3/637.3 kB 3.5 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 3.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [openmeteo-requests]iquests]uture]
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 2.3 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 4.1 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [requests-cache]m [pandas]]
Note: you may need to restart the kernel to use updated packages.


In [8]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"hourly": ["temperature_2m", "rain", "wind_speed_10m"],
	"models": "best_match",
	"current": ["rain", "temperature_2m"],
	"timezone": "Asia/Bangkok",
	"past_days": 31,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process current data. The order of variables needs to be the same as requested.
current = response.Current()
current_rain = current.Variables(0).Value()
current_temperature_2m = current.Variables(1).Value()

print(f"\nCurrent time: {current.Time()}")
print(f"Current rain: {current_rain}")
print(f"Current temperature_2m: {current_temperature_2m}")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_rain = hourly.Variables(1).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(2).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["rain"] = hourly_rain
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 52.52000045776367°N 13.419998168945312°E
Elevation: 38.0 m asl
Timezone: b'Asia/Bangkok'b'GMT+7'
Timezone difference to GMT+0: 25200s

Current time: 1787930100
Current rain: 0.0
Current temperature_2m: 25.59549903869629

Hourly data
                          date  temperature_2m  rain  wind_speed_10m
0   2026-07-28 00:00:00+07:00       20.395500   0.0       16.595179
1   2026-07-28 01:00:00+07:00       19.645500   0.0       13.722565
2   2026-07-28 02:00:00+07:00       18.995501   0.0       12.979984
3   2026-07-28 03:00:00+07:00       18.245501   0.0       12.429127
4   2026-07-28 04:00:00+07:00       17.245501   0.0       12.324414
..                        ...             ...   ...             ...
907 2026-09-03 19:00:00+07:00       21.369501   0.0       15.137133
908 2026-09-03 20:00:00+07:00       21.169500   0.0       14.512064
909 2026-09-03 21:00:00+07:00       20.419500   0.0       13.779114
910 2026-09-03 22:00:00+07:00       19.469501   0.0       13.009903
911 2